In [ ]:
from flask import Flask, request, jsonify
import joblib
import pyodbc
import pandas as pd
import sys

app = Flask(__name__)

try:
    model = joblib.load('fitness_model.pkl')
    print(">>> Model başarıyla yüklendi.")
except FileNotFoundError:
    print(">>> HATA: fitness_model.pkl dosyası bulunamadı!")
    sys.exit(1)
    
def get_sql_connection():
    conn_str = (
        r'Driver={SQL Server};'
        r'Server=GHOST;'
        r'Database=FitnessAI;'
        r'Trusted_Connection=yes;'
    )
    return pyodbc.connect(conn_str)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        
        input_dict = {
            'Yas': [int(data.get('Yas', 0))],
            'Boy': [int(data.get('Boy', 0))],
            'Kilo': [float(data.get('Kilo', 0))],
            'VKI': [float(data.get('VKI', 0))],
            'Sakatlik_Siddeti': [int(data.get('Sakatlik_Siddeti', 0))],
            'Spor_Gecmisi_Yil': [int(data.get('Spor_Gecmisi_Yil', 0))]
        }
        
        input_df = pd.DataFrame(input_dict)
        
        prediction_label = model.predict(input_df)[0]

        conn = get_sql_connection()
        cursor = conn.cursor()
        
        query = """
            SELECT Program_Tam_Adi, Odak_Noktasi, Hareketler, Beslenme_Tipi 
            FROM Program_Detaylari 
            WHERE Program_Etiketi = ?
        """
        cursor.execute(query, (prediction_label,))
        row = cursor.fetchone()
        
        if row:
            
            result = {
                "status": "success",
                "model_tahmini": prediction_label,
                "program_detaylari": {
                    "isim": row[0],
                    "odak": row[1],
                    "egzersizler": row[2],
                    "beslenme": row[3]
                }
            }
        else:
            result = {
                "status": "partial_success",
                "model_tahmini": prediction_label,
                "message": "Model tahmin yaptı ancak SQL'de detaylı açıklama bulunamadı."
            }
            
        conn.close()
        return jsonify(result)

    except Exception as e:
        return jsonify({"status": "error", "message": str(e)})

if __name__ == '__main__':
    app.run(debug=True, port=5000, use_reloader=False)

>>> Model başarıyla yüklendi.
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
